# Ookla vs NDT7 — Thailand Province-Level Comparison

**Overlap period:** 2023-Q4 – 2025-Q1 (6 quarters)  
**Unit of analysis:** province × quarter  
**Sources:** Ookla Speedtest Intelligence (fixed broadband) · M-Lab NDT7 (broadband)

> Run `ndt7_eda.ipynb` §23 and `ookla_eda.ipynb` export cell first to generate CSVs.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import geopandas as gpd
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 100,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Paths ──────────────────────────────────────────────────────────
OOKLA_CSV  = '../../data/exports/ookla_province_quarterly.csv'
NDT7_CSV   = '../../data/exports/ndt7_province_quarterly_broadband.csv'
GEOJSON    = '../../data/geo/thailand_provinces.geojson'
PROV_REF   = '../../data/reference/province_reference.csv'

# ── Overlap quarters (NDT7 range) ──────────────────────────────────
OVERLAP = ['2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1']

# ── Colors ─────────────────────────────────────────────────────────
C_OOKLA = '#e63946'
C_NDT7  = '#457b9d'
REGION_COLORS = {
    'Bangkok & Vicinity': '#e63946',
    'Eastern':            '#f4a261',
    'Central':            '#2a9d8f',
    'Western':            '#8338ec',
    'Northern':           '#457b9d',
    'Southern':           '#ffb703',
    'Northeastern':       '#6d6875',
}
TIER_COLORS = {1: '#e63946', 2: '#f4a261', 3: '#2a9d8f', 4: '#6d6875'}

print('Config loaded ✓')

## 1 — Load & Normalize Both Datasets

In [ ]:
for p, name in [(OOKLA_CSV, 'Ookla CSV'), (NDT7_CSV, 'NDT7 CSV')]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'{name} not found at {p}.\nRun the export cell in the source notebook first.')

# ── Load Ookla ─────────────────────────────────────────────────────
ookla_raw = pd.read_csv(OOKLA_CSV)
ookla_raw['source'] = 'Ookla'
ookla = ookla_raw[ookla_raw['quarter'].isin(OVERLAP)].copy()

# ── Load NDT7 ──────────────────────────────────────────────────────
ndt7_raw = pd.read_csv(NDT7_CSV)
ndt7_raw['source'] = 'NDT7'
ndt7 = ndt7_raw[ndt7_raw['year_q'].isin(OVERLAP)].copy()
ndt7 = ndt7.rename(columns={'year_q': 'quarter', 'avg_d': 'avg_d_mbps', 'med_d': 'med_d_mbps',
                              'avg_u': 'avg_u_mbps', 'med_u': 'med_u_mbps', 'avg_lat': 'avg_lat_ms_wt'})

# ── Common columns ─────────────────────────────────────────────────
COMMON = ['source', 'province', 'quarter', 'avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms_wt',
          'total_tests', 'n_tiles', 'is_reliable', 'region', 'internet_tier',
          'pop_2024', 'gdp_per_capita_thb_2021', 'density_per_km2']

def align(df):
    for c in COMMON:
        if c not in df.columns:
            df[c] = np.nan
    return df[COMMON].copy()

ookla = align(ookla)
ndt7  = align(ndt7)

print(f'Ookla overlap:  {len(ookla):,} rows | provinces={ookla.province.nunique()} | quarters={ookla.quarter.nunique()}')
print(f'NDT7  overlap:  {len(ndt7):,}  rows | provinces={ndt7.province.nunique()} | quarters={ndt7.quarter.nunique()}')

## 2 — Reliability Overview

In [ ]:
# Per-source reliability summary
for src, df in [('Ookla', ookla), ('NDT7', ndt7)]:
    r = df.groupby('quarter')['is_reliable'].agg(['sum', 'count'])
    r['pct'] = r['sum'] / r['count'] * 100
    print(f'=== {src} reliable province-quarters ===')
    print(r.rename(columns={'sum': 'reliable', 'count': 'total'}).to_string())
    print()

# Provinces reliable in BOTH sources (per quarter) → basis for statistical tests
both_reliable = (
    ookla[ookla['is_reliable']].merge(
        ndt7[ndt7['is_reliable']],
        on=['province', 'quarter'],
        suffixes=('_ookla', '_ndt7')
    )
)
print(f'Province-quarters reliable in BOTH: {len(both_reliable)} / {len(OVERLAP)*77}')

In [ ]:
# Test count distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (src, df, color) in zip(axes, [('Ookla', ookla, C_OOKLA), ('NDT7', ndt7, C_NDT7)]):
    prov_tests = df.groupby('province')['total_tests'].sum()
    ax.hist(prov_tests, bins=30, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(prov_tests.median(), color='black', ls='--', lw=1.5,
               label=f'Median {prov_tests.median():,.0f}')
    ax.set_xlabel('Total Tests (all overlap quarters)')
    ax.set_title(f'{src} — Tests per Province')
    ax.legend()

plt.suptitle('Test Volume Distribution — Ookla vs NDT7 Broadband', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 — Parallel EDA: Speed Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (src, df, color) in enumerate([
    ('Ookla Fixed', ookla, C_OOKLA),
    ('NDT7 Broadband', ndt7, C_NDT7),
]):
    for col, (metric, label) in enumerate([
        ('avg_d_mbps', 'Download (Mbps)'),
        ('avg_u_mbps', 'Upload (Mbps)'),
    ]):
        ax = axes[row, col]
        vals = df[metric].dropna()
        ax.hist(vals, bins=40, color=color, alpha=0.82, edgecolor='none')
        med = vals.median()
        ax.axvline(med, color='black', ls='--', lw=1.5, label=f'Median {med:.1f}')
        ax.set_xlabel(label)
        ax.set_title(f'{src} — {label}')
        ax.legend()

plt.suptitle('Speed Distributions: Province-Level Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 — Province Rankings Side-by-Side

In [ ]:
# Aggregate across all overlap quarters (test-weighted mean)
def province_mean(df):
    return (
        df.dropna(subset=['avg_d_mbps', 'total_tests'])
        .groupby(['province', 'region', 'internet_tier'])
        .apply(lambda g: pd.Series({
            'avg_dl': np.average(g['avg_d_mbps'], weights=g['total_tests']),
            'avg_ul': np.average(g['avg_u_mbps'].fillna(0), weights=g['total_tests']),
            'total_tests': g['total_tests'].sum(),
        }), include_groups=False)
        .reset_index()
    )

prov_ookla = province_mean(ookla)
prov_ndt7  = province_mean(ndt7)

# Sort by Ookla DL; show both
prov_merged = prov_ookla.merge(prov_ndt7[['province', 'avg_dl', 'avg_ul']],
                               on='province', how='inner', suffixes=('_ookla', '_ndt7'))
prov_merged = prov_merged.sort_values('avg_dl_ookla', ascending=True)

fig, ax = plt.subplots(figsize=(14, 22))
y = np.arange(len(prov_merged))
ax.barh(y + 0.2, prov_merged['avg_dl_ookla'], 0.38, color=C_OOKLA, alpha=0.85, label='Ookla')
ax.barh(y - 0.2, prov_merged['avg_dl_ndt7'],  0.38, color=C_NDT7,  alpha=0.85, label='NDT7')
ax.set_yticks(y)
ax.set_yticklabels(prov_merged['province'], fontsize=8)
ax.set_xlabel('Mean Download Speed (Mbps)')
ax.set_title('Province Download Speed — Ookla vs NDT7 (sorted by Ookla)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5 — Heatmaps: Province × Quarter

In [ ]:
sort_order = prov_merged.sort_values('avg_dl_ookla', ascending=False)['province'].tolist()

for src, df, cmap in [('Ookla', ookla, 'YlOrRd'), ('NDT7', ndt7, 'YlGnBu')]:
    pivot = df.pivot_table(index='province', columns='quarter', values='avg_d_mbps', aggfunc='mean')
    pivot = pivot.reindex([p for p in sort_order if p in pivot.index])
    pivot = pivot.reindex(columns=OVERLAP)

    fig, ax = plt.subplots(figsize=(12, 18))
    sns.heatmap(pivot, cmap=cmap, annot=False, cbar_kws={'label': 'Mean DL (Mbps)', 'shrink': 0.5}, ax=ax)
    ax.set_title(f'{src} — Mean Download Speed Heatmap (Province × Quarter)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Quarter'); ax.set_ylabel('Province')
    plt.tight_layout()
    plt.show()

## 6 — Regional Comparison

In [ ]:
region_order = ['Bangkok & Vicinity', 'Eastern', 'Central', 'Western', 'Southern', 'Northern', 'Northeastern']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (src, df, color) in zip(axes, [('Ookla', ookla, C_OOKLA), ('NDT7', ndt7, C_NDT7)]):
    data = [df[df['region'] == r]['avg_d_mbps'].dropna().values for r in region_order]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='white', linewidth=2))
    for patch, region in zip(bp['boxes'], region_order):
        patch.set_facecolor(REGION_COLORS.get(region, 'gray'))
        patch.set_alpha(0.85)
    ax.set_xticklabels([r.replace(' & ', '\n& ') for r in region_order], fontsize=8)
    ax.set_ylabel('Mean Download (Mbps)')
    ax.set_title(f'{src} Download Speed by Region')

plt.suptitle('Regional Download Speed Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7 — Trend Lines

In [ ]:
HIGHLIGHT = {
    'Top performers':   ['Nonthaburi', 'Pathum Thani', 'Samut Prakan', 'Chon Buri', 'Bangkok Metropolis'],
    'Low performers':   ['Mae Hong Son', 'Satun', 'Tak', 'Phangnga', 'Ranong'],
    'Provincial hubs':  ['Phuket', 'Chiang Mai', 'Khon Kaen', 'Nakhon Ratchasima', 'Udon Thani'],
}

fig, axes = plt.subplots(3, 2, figsize=(18, 18), sharey='row')

for row, (group_name, provinces) in enumerate(HIGHLIGHT.items()):
    for col, (src, df, color) in enumerate([('Ookla', ookla, C_OOKLA), ('NDT7', ndt7, C_NDT7)]):
        ax = axes[row, col]
        for prov in provinces:
            s = df[df['province'] == prov].sort_values('quarter')
            if s.empty: continue
            ax.plot(s['quarter'], s['avg_d_mbps'], 'o-', lw=2, label=prov)
        ax.set_ylabel('Mean DL (Mbps)')
        ax.set_title(f'{src} — {group_name}')
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=30)

plt.suptitle('Download Speed Trends (2023-Q4 – 2025-Q1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 — Growth Analysis

In [ ]:
first_q, last_q = OVERLAP[0], OVERLAP[-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 18))

for ax, (src, df, color) in zip(axes, [('Ookla', ookla, C_OOKLA), ('NDT7', ndt7, C_NDT7)]):
    f_vals = df[df['quarter'] == first_q].set_index('province')['avg_d_mbps']
    l_vals = df[df['quarter'] == last_q].set_index('province')['avg_d_mbps']
    growth = ((l_vals - f_vals) / f_vals * 100).dropna().sort_values(ascending=True)
    growth_df = growth.reset_index(name='pct')
    growth_df = growth_df.merge(
        df[['province', 'region']].drop_duplicates(), on='province', how='left'
    )
    colors = [REGION_COLORS.get(r, 'gray') for r in growth_df['region']]
    ax.barh(growth_df['province'], growth_df['pct'], color=colors, edgecolor='none', height=0.7)
    ax.axvline(0, color='black', lw=0.8)
    avg = growth_df['pct'].mean()
    ax.axvline(avg, color='red', ls='--', lw=1.2, label=f'Avg {avg:.1f}%')
    ax.set_xlabel('DL Growth (%)')
    ax.set_title(f'{src}\n{first_q} → {last_q}', fontsize=12, fontweight='bold')
    ax.legend()

plt.suptitle('Province Download Speed Growth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9 — Direct Comparison: Ookla DL vs NDT7 DL per Province

*Scatter at province level. Each dot = one province × quarter. Reliable set highlighted.*

In [ ]:
# Merge on province × quarter
comp = ookla.merge(
    ndt7[['province', 'quarter', 'avg_d_mbps', 'med_d_mbps', 'is_reliable']],
    on=['province', 'quarter'],
    suffixes=('_ookla', '_ndt7'),
    how='inner'
)
comp['both_reliable'] = comp['is_reliable_ookla'] & comp['is_reliable_ndt7']

print(f'Matched province-quarters: {len(comp)}')
print(f'Both reliable: {comp["both_reliable"].sum()}')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, ndt7_col, ndt7_lbl in [
    (axes[0], 'avg_d_mbps_ndt7', 'NDT7 Mean DL'),
    (axes[1], 'med_d_mbps',      'NDT7 Median DL'),
]:
    # All points — grey
    ax.scatter(comp['avg_d_mbps_ookla'], comp[ndt7_col],
               c='lightgray', s=25, alpha=0.5, label='All')
    # Reliable — colored by region
    rel = comp[comp['both_reliable']]
    colors = [REGION_COLORS.get(r, 'gray') for r in rel['region']]
    ax.scatter(rel['avg_d_mbps_ookla'], rel[ndt7_col],
               c=colors, s=55, alpha=0.85, edgecolors='white', lw=0.5, label='Reliable')

    # Diagonal (perfect agreement)
    lim = max(comp['avg_d_mbps_ookla'].max(), comp[ndt7_col].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.4, label='1:1')

    # OLS on reliable set
    if len(rel) > 5:
        slope, intc, r_val, p_val, _ = scipy_stats.linregress(
            rel['avg_d_mbps_ookla'].dropna(), rel[ndt7_col].dropna()
        )
        x_fit = np.linspace(rel['avg_d_mbps_ookla'].min(), rel['avg_d_mbps_ookla'].max(), 100)
        ax.plot(x_fit, slope * x_fit + intc, color='crimson', lw=1.5,
                label=f'OLS r={r_val:.2f}, p={p_val:.3f}')

    ax.set_xlabel('Ookla Mean DL (Mbps)')
    ax.set_ylabel(f'{ndt7_lbl} (Mbps)')
    ax.set_title(f'Ookla vs {ndt7_lbl}')
    ax.legend(fontsize=8)

plt.suptitle('Ookla Fixed vs NDT7 Broadband Download Speed — Province × Quarter', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10 — Gap Analysis: NDT7 − Ookla

In [ ]:
comp['gap'] = comp['avg_d_mbps_ndt7'] - comp['avg_d_mbps_ookla']
comp['gap_pct'] = comp['gap'] / comp['avg_d_mbps_ookla'] * 100

rel = comp[comp['both_reliable']].copy()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Gap histogram
axes[0].hist(rel['gap'], bins=30, color='#6d6875', edgecolor='none', alpha=0.85)
axes[0].axvline(0, color='black', lw=1)
axes[0].axvline(rel['gap'].mean(), color='red', ls='--', lw=1.5, label=f"Mean gap {rel['gap'].mean():.1f} Mbps")
axes[0].set_xlabel('NDT7 − Ookla (Mbps)')
axes[0].set_title('Gap Distribution (Reliable)')
axes[0].legend()

# Gap % histogram
axes[1].hist(rel['gap_pct'], bins=30, color='#457b9d', edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='black', lw=1)
axes[1].axvline(rel['gap_pct'].mean(), color='red', ls='--', lw=1.5,
                label=f"Mean {rel['gap_pct'].mean():.1f}%")
axes[1].set_xlabel('Gap as % of Ookla speed')
axes[1].set_title('Relative Gap % (Reliable)')
axes[1].legend()

# Gap by tier
tier_gap = rel.groupby('internet_tier')['gap'].agg(['mean', 'median', 'count'])
axes[2].bar(tier_gap.index.astype(str), tier_gap['mean'],
            color=[TIER_COLORS.get(t, 'gray') for t in tier_gap.index], alpha=0.85)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_xlabel('Internet Tier')
axes[2].set_ylabel('Mean Gap (Mbps)')
axes[2].set_title('Mean Gap by Tier')
for i, (t, row) in enumerate(tier_gap.iterrows()):
    axes[2].text(i, row['mean'] + 0.5, f"n={row['count']:.0f}", ha='center', fontsize=9)

plt.suptitle('NDT7 − Ookla Download Speed Gap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Gap Summary (Reliable province-quarters) ===')
print(f'  Mean gap  : {rel["gap"].mean():+.2f} Mbps')
print(f'  Median gap: {rel["gap"].median():+.2f} Mbps')
print(f'  Std gap   : {rel["gap"].std():.2f} Mbps')
print(f'  NDT7 > Ookla: {(rel["gap"] > 0).mean():.1%} of province-quarters')

In [ ]:
# Gap bar chart by province (averaged across quarters)
prov_gap = rel.groupby(['province', 'region'])['gap'].mean().reset_index().sort_values('gap')

fig, ax = plt.subplots(figsize=(12, 18))
colors = [REGION_COLORS.get(r, 'gray') for r in prov_gap['region']]
bars = ax.barh(prov_gap['province'], prov_gap['gap'], color=colors, edgecolor='none', height=0.7)
ax.axvline(0, color='black', lw=1)
ax.axvline(prov_gap['gap'].mean(), color='red', ls='--', lw=1.2,
           label=f"Mean {prov_gap['gap'].mean():+.1f} Mbps")
for bar, val in zip(bars, prov_gap['gap']):
    ax.text(bar.get_width() + (0.3 if val >= 0 else -0.5),
            bar.get_y() + bar.get_height()/2, f'{val:+.1f}', va='center', fontsize=7.5)
patches = [mpatches.Patch(color=v, label=k) for k, v in REGION_COLORS.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_xlabel('NDT7 − Ookla Mean DL (Mbps)')
ax.set_title('Province-Level Gap: NDT7 Broadband minus Ookla Fixed\n(positive = NDT7 faster)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11 — Statistical Tests (Reliable Set)

In [ ]:
rel = comp[comp['both_reliable']].dropna(subset=['avg_d_mbps_ookla', 'avg_d_mbps_ndt7'])

# Wilcoxon signed-rank test
stat_w, p_w = scipy_stats.wilcoxon(rel['avg_d_mbps_ndt7'], rel['avg_d_mbps_ookla'])

# Paired t-test
stat_t, p_t = scipy_stats.ttest_rel(rel['avg_d_mbps_ndt7'], rel['avg_d_mbps_ookla'])

# Cohen's d
d = rel['gap'].mean() / rel['gap'].std()

# Pearson r (agreement)
r_val, r_p = scipy_stats.pearsonr(rel['avg_d_mbps_ookla'], rel['avg_d_mbps_ndt7'])

# Spearman (rank agreement)
rho, rho_p = scipy_stats.spearmanr(rel['avg_d_mbps_ookla'], rel['avg_d_mbps_ndt7'])

print('=' * 60)
print(f'  Statistical Tests — NDT7 vs Ookla Download Speed')
print(f'  n = {len(rel)} reliable province × quarter pairs')
print('=' * 60)
print(f'  Wilcoxon signed-rank  : W={stat_w:.1f},  p={p_w:.4f}')
print(f'  Paired t-test         : t={stat_t:.2f},  p={p_t:.4f}')
print(f"  Cohen's d             : {d:+.3f} ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'})")
print(f'  Pearson r (agreement) : {r_val:.3f}  (p={r_p:.4f})')
print(f'  Spearman rho          : {rho:.3f}  (p={rho_p:.4f})')
print('=' * 60)

## 12 — Explanatory Analysis: Why Is There a Gap?

*MacMillan et al. (2023): NDT7 underreports speed by 12–56% in high-latency environments.*

In [ ]:
# Need latency from NDT7 for gap vs latency analysis
rel_lat = rel.dropna(subset=['avg_lat_ms_wt_ndt7', 'gap'])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# A. Gap vs NDT7 latency
ax = axes[0]
if len(rel_lat) > 0:
    ax.scatter(rel_lat['avg_lat_ms_wt_ndt7'], rel_lat['gap'],
               c=[REGION_COLORS.get(r, 'gray') for r in rel_lat['region']],
               s=45, alpha=0.7, edgecolors='white', lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    if len(rel_lat) > 5:
        s, i, r, p, _ = scipy_stats.linregress(rel_lat['avg_lat_ms_wt_ndt7'], rel_lat['gap'])
        x_fit = np.linspace(rel_lat['avg_lat_ms_wt_ndt7'].min(), rel_lat['avg_lat_ms_wt_ndt7'].max(), 100)
        ax.plot(x_fit, s * x_fit + i, 'k--', lw=1.5, label=f'r={r:.2f}, p={p:.3f}')
        ax.legend()
ax.set_xlabel('NDT7 Mean Latency (ms)')
ax.set_ylabel('Gap (NDT7 − Ookla, Mbps)')
ax.set_title('Latency vs Speed Gap\n(high RTT → NDT7 underreport?)')

# B. Gap vs Ookla speed level (faster provinces — bigger gap?)
ax2 = axes[1]
ax2.scatter(rel['avg_d_mbps_ookla'], rel['gap'],
            c=[REGION_COLORS.get(r, 'gray') for r in rel['region']],
            s=45, alpha=0.7, edgecolors='white', lw=0.5)
ax2.axhline(0, color='black', lw=0.8)
if len(rel) > 5:
    s2, i2, r2, p2, _ = scipy_stats.linregress(rel['avg_d_mbps_ookla'], rel['gap'])
    x2 = np.linspace(rel['avg_d_mbps_ookla'].min(), rel['avg_d_mbps_ookla'].max(), 100)
    ax2.plot(x2, s2 * x2 + i2, 'k--', lw=1.5, label=f'r={r2:.2f}, p={p2:.3f}')
    ax2.legend()
ax2.set_xlabel('Ookla Mean DL (Mbps)')
ax2.set_ylabel('Gap (NDT7 − Ookla, Mbps)')
ax2.set_title('Ookla Speed Level vs Gap\n(selection bias check)')

# C. Gap by region
ax3 = axes[2]
reg_gap = rel.groupby('region')['gap'].agg(['mean', 'std', 'count']).reset_index()
reg_gap = reg_gap.sort_values('mean')
colors_r = [REGION_COLORS.get(r, 'gray') for r in reg_gap['region']]
ax3.barh(reg_gap['region'], reg_gap['mean'], color=colors_r, alpha=0.85, edgecolor='none')
ax3.axvline(0, color='black', lw=0.8)
for i, (_, row) in enumerate(reg_gap.iterrows()):
    ax3.text(row['mean'] + 0.3, i, f"+/-{row['std']:.1f} (n={row['count']:.0f})", va='center', fontsize=8)
ax3.set_xlabel('Mean Gap (Mbps)')
ax3.set_title('Gap by Region')

plt.suptitle('Explanatory Analysis: What Drives the Ookla–NDT7 Gap?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 13 — Coverage Bias: Who Is Measured?

*Different user bases → different provinces over/under-represented.*

In [ ]:
# Tests per province — Ookla vs NDT7 (normalized by pop if available)
ookla_cov = ookla.groupby('province').agg(ookla_tests=('total_tests', 'sum'), pop=('pop_2024', 'first')).reset_index()
ndt7_cov  = ndt7.groupby('province').agg(ndt7_tests=('total_tests', 'sum')).reset_index()
cov = ookla_cov.merge(ndt7_cov, on='province', how='outer').fillna(0)
cov['ookla_per1k'] = cov['ookla_tests'] / cov['pop'] * 1000
cov['ndt7_per1k']  = cov['ndt7_tests']  / cov['pop'] * 1000
cov['test_ratio']  = np.log1p(cov['ndt7_tests']) / np.log1p(cov['ookla_tests'])  # log ratio

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter: Ookla coverage vs NDT7 coverage
axes[0].scatter(cov['ookla_per1k'], cov['ndt7_per1k'], s=60, alpha=0.8,
                color='#6d6875', edgecolors='white', lw=0.5)
for _, row in cov.nlargest(10, 'ookla_per1k').iterrows():
    axes[0].annotate(row['province'], (row['ookla_per1k'], row['ndt7_per1k']), fontsize=7)
axes[0].set_xlabel('Ookla Tests per 1,000 Pop')
axes[0].set_ylabel('NDT7 Tests per 1,000 Pop')
axes[0].set_title('Coverage Comparison: Ookla vs NDT7')

# Bar: log-scale test ratio
cov_s = cov.sort_values('test_ratio')
axes[1].barh(cov_s['province'], cov_s['test_ratio'], color='#457b9d', alpha=0.8, edgecolor='none')
axes[1].axvline(1.0, color='black', ls='--', lw=1, label='Equal log-coverage')
axes[1].set_xlabel('Log(NDT7 tests) / Log(Ookla tests)')
axes[1].set_title('Relative Coverage: NDT7 vs Ookla\n(>1 = NDT7 proportionally more)')
axes[1].legend()

plt.suptitle('Test Coverage Bias — Who Gets Measured?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 14 — Summary Table

In [ ]:
summary_cols = ['province', 'region', 'internet_tier',
                'avg_d_mbps_ookla', 'avg_d_mbps_ndt7', 'gap', 'gap_pct', 'both_reliable']

prov_summary = (
    comp.groupby(['province', 'region', 'internet_tier'])
    .agg(
        avg_dl_ookla=('avg_d_mbps_ookla', 'mean'),
        avg_dl_ndt7=('avg_d_mbps_ndt7', 'mean'),
        avg_gap=('gap', 'mean'),
        avg_gap_pct=('gap_pct', 'mean'),
        n_quarters=('quarter', 'count'),
        reliable_quarters=('both_reliable', 'sum'),
    )
    .reset_index()
    .sort_values('avg_gap', ascending=False)
    .round(2)
)

print(prov_summary.to_string(index=False))